In [0]:
%sql
CREATE CATALOG IF NOT EXISTS gridpulse_dev;
COMMENT ON CATALOG gridpulse_dev IS 'Raw data ingested from Elexon BMRS API';

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS gridpulse_dev.landing;
CREATE SCHEMA IF NOT EXISTS gridpulse_dev.bronze;
CREATE SCHEMA IF NOT EXISTS gridpulse_dev.silver;
CREATE SCHEMA IF NOT EXISTS gridpulse_dev.gold;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS
gridpulse_dev.landing.raw_landing;

In [0]:
%sql
SHOW SCHEMAS IN gridpulse_dev;

In [0]:
display(dbutils.fs.ls(
    "/Volumes/gridpulse_dev/landing/raw_landing/"
))

In [0]:
import requests

url = (
    "https://data.elexon.co.uk/bmrs/api/v1/"
    "datasets/FUELHH/stream"
)

params = {
    "settlementDateFrom": "2026-11-16",
    "settlementDateTo": "2026-11-16"
}

response = requests.get(
    url,
    params=params,
    timeout=30
)

response.raise_for_status()
data = response.json()

print("HTTP status:", response.status_code)
print("Response type:", type(data).__name__)
print("Records:", len(data))
print("Sample:", data[0] if data else None)

In [0]:
from pathlib import Path
from datetime import datetime, timezone

landing = Path(
    "/Volumes/gridpulse_dev/landing/"
    "raw_landing/elexon/fuelhh"
)

landing.mkdir(parents=True, exist_ok=True)

run_id = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%SZ"
)

raw_path = landing / f"fuelhh_{run_id}.json"

raw_path.write_bytes(response.content)

print("Saved:", raw_path)
print("Size:", raw_path.stat().st_size, "bytes")

In [0]:
df_elexon = (
    spark.read
    .option("multiLine", True)
    .json(str(raw_path))
)

df_elexon.printSchema()

display(df_elexon.limit(10))

print("Total rows:", df_elexon.count())

In [0]:
from pyspark.sql import functions as F

df_bronze = (
    df_elexon
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.lit(str(raw_path)))
    .withColumn("_source_system", F.lit("ELEXON"))
)

table_name = "gridpulse_dev.bronze.elexon_generation"

(
    df_bronze.write
    .format("delta")
    .mode("errorifexists")
    .saveAsTable(table_name)
)

In [0]:
%sql
SELECT COUNT(*)
FROM gridpulse_dev.bronze.elexon_generation;